In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [2]:
csv_filename_train = "../CleanData/full_train_05_09.csv"
df_train = pd.read_csv(csv_filename_train)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72967 entries, 0 to 72966
Data columns (total 88 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   site                  72967 non-null  object 
 1   daynight              72967 non-null  int64  
 2   attendance            72967 non-null  int64  
 3   temp                  72967 non-null  float64
 4   windspeed             72967 non-null  float64
 5   Home_Away             72967 non-null  int64  
 6   left_on_base          72967 non-null  int64  
 7   bat_doubles           72967 non-null  int64  
 8   bat_triples           72967 non-null  int64  
 9   bat_home_runs         72967 non-null  int64  
 10  bat_stolen_bases      72967 non-null  int64  
 11  passed_balls_allowed  72967 non-null  int64  
 12  errors_committed      72967 non-null  int64  
 13  stolen_bases          72967 non-null  int64  
 14  yearID                72967 non-null  int64  
 15  salary             

In [3]:
csv_filename_train = "../CleanData/full_test_05_09.csv"
df_test = pd.read_csv(csv_filename_train)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18242 entries, 0 to 18241
Data columns (total 88 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   site                  18242 non-null  object 
 1   daynight              18242 non-null  int64  
 2   attendance            18242 non-null  int64  
 3   temp                  18242 non-null  float64
 4   windspeed             18242 non-null  float64
 5   Home_Away             18242 non-null  int64  
 6   left_on_base          18242 non-null  int64  
 7   bat_doubles           18242 non-null  int64  
 8   bat_triples           18242 non-null  int64  
 9   bat_home_runs         18242 non-null  int64  
 10  bat_stolen_bases      18242 non-null  int64  
 11  passed_balls_allowed  18242 non-null  int64  
 12  errors_committed      18242 non-null  int64  
 13  stolen_bases          18242 non-null  int64  
 14  yearID                18242 non-null  int64  
 15  salary             

#### model with XGBoost

In [4]:
df_test = df_test.drop(columns=['site'])
df_train = df_train.drop(columns=['site'])

y_train = df_train['score_diff']
df_train = df_train.drop(columns=['score_diff'])
y_test = df_test['score_diff']
df_test = df_test.drop(columns=['score_diff'])


reg_xgb = XGBRegressor(max_depth = 3, learning_rate = 0.1, n_estimators = 100, n_jobs=2, objective='reg:squarederror', random_state=500, enable_categorical=True)
reg_xgb.fit(df_train, y_train)

pred_xgb = reg_xgb.predict(df_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))

print('RMSE Xgboost: {:.3f}'.format(rmse_xgb))

RMSE Xgboost: 3.654


In [6]:
# try to predict a winner based on score_diff
df_xgboost_pred = df_test.assign(
    score_diff_pred = pred_xgb
)
correct_pred = 0
incorrect_pred = 0
for i, row in df_xgboost_pred.iterrows():
    if row['score_diff_pred'] > 0 and y_test[i] > 0:
        correct_pred += 1
    elif row['score_diff_pred'] < 0 and y_test[i] < 0:
        correct_pred += 1
    else:
        incorrect_pred += 1

print("Correct Predictions:", correct_pred, "Incorrect predictions:", incorrect_pred)
print("Percentage of correct pred:", correct_pred / (correct_pred + incorrect_pred))

Correct Predictions: 12694 Incorrect predictions: 5548
Percentage of correct pred: 0.6958666812849468
